In [8]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pyarrow as pa

In [ ]:
df_features = pd.read_parquet("res//procesed/df_features.parquet")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)   
display(df_features.head(1))

,fecha_fase,IDMOTask,smOrdenId,smOrdenNme,smFaseNme,Description,IDUsualTask,MaquinaDominante,NombreMaquina,CodigoGrupo,NombreGrupo,smMaterialId,Descripción Artículo,CodProductLine,Nombre Línea,CodProductFamily,Nombre Família,ProductEntry,MaterialExit,Quantity,tiempo_real_efectivo,tiempo_previsto,ratio
0,2025-10-02 03:12:05.010,ee76d685-a4cc-47d4-b418-6e33d2ae2921,OF2509545,Portacápsulas F704501049704(150713),TPP,Mecanitzat al Torn Puma Petit,29783995-2f56-48be-9664-461e6f0b402a,TN09,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,TPU,Tornos Puma Cobre,PE1JNCX0012,Elektrodenaufnamhe F704501049704(DS086228),P,Portalectrodes,PE,Portalectrodes,0.0,0.0,3.0,41.76667,21.0,1.988889


## Split de Datos
Teniendo en cuenta la naturaleza de estos datos es necesario plantear un split temporal. Para ello el campo `fecha_fase` permite identificar la fecha en que se empezó la fase.      

Una Orden de Fabricación puede tener distintas Fases, para este split es importante que las Fases de una misma Orden no se separen. Es decir, las Fases de una misma Orden no pueden aparecer en distitnas particiones.     

Jacky recomienda: El ajuste de hiperparámetros se hace solo dentro de train, con TimeSeriesSplit. El test se toca una única vez, al final     

Análisis de Generalización: dentro del test se marcan las fases de órdenes o combinaciones máquina-fase no vistas en train y se reporta su error por separado.

## Baselines: un número
- **B0: ratio = 1** -> es la planificación definida: El ERP acierta.    
- **B1- mediana: ratio** = mediana del train -> si todos los datos tienen una desviación sistematica, cualquier modelo predicirá el sesgo B0: El ERP tiene un sesgo fijo.  
    > mediana suponiendo que trabajemos con MAE, sino será otra la constante.     

- **B1 - media: ratio** = media del train -> si todos los datos tienen una desviación sistematica, cualquier modelo predicirá el sesgo B0: El ERP tiene un sesgo
    > media suponiendo que trabajemos con RMSE, sino será otra constante.

B0 y B1 ignoran por completo las variables de entrada. No saben qué máquina se usa, ni la fase, ni la cantidad.    

> Importante: la mediana se calcula solo con el train. Si la calculas con todo el dataset, el baseline "conoce" el test. Sería leakage, igual que en un modelo.

**Constante B1**: A favor de la **mediana** y no la **media**: el ratio es asimétrico (cola larga a la derecha, por eso te planteas log(ratio)). Unas pocas fases con ratio 6 u 8 inflan la media, mientras que la mediana representa mejor a la "fase típica".

## Baselines: Una tabla de consulta (referencia intermedia)
Constante general o por maquian y linea de producto
- **B2 - media o mediana ratio por grupo = cosntante del train**
    > Deben elegirse grupos con suficientes fases en train
Ahora **B2** ya utilizará información de las variables. 

 B0 (ERP) ──► B1 (corrijo sesgo global) ──► B2 (tabla por grupo) ──► Modelo ML        
    │                 │                              │                    │                            
    └ ¿ERP sesgado? ──┘                              │                    │       
                      └ ¿las variables aportan algo? ┘                    │          
                                                     └ ¿compensa el ML? ──┘        

Si los modelos no superan B2: hay señal, pero una tabla simple la captura casi toda. Eso también es un resultado válido y defendible.

In [ ]:
# guardar
# df_features.to_parquet("res//procesed/df_features.parquet")

Apenas modificamos el dataset, ya que cada modelo precisara de su propio feature engeniering. 